In [1]:
from google.colab import drive
import torch
import os
import glob
import numpy as np
import cv2

In [2]:
#Mount Google Drive and navigate to the assignment directory
drive.mount('/content/drive')
%cd /content/drive/MyDrive/assignments/Video_action_recognition
!git pull
%pip install -q -r requirements.txt
%env PYTHONPATH=src
%env UCF50_N_FRAMES=16
with open("/content/drive/MyDrive/credentials/wandb_api.txt") as f:
    os.environ["WANDB_API_KEY"] = f.read().strip()
print("W&B key loaded:", len(os.environ.get("WANDB_API_KEY", "")) > 0)
# Check for PyTorch and CUDA availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/assignments/Video_action_recognition
Already up to date.
env: PYTHONPATH=src
env: UCF50_N_FRAMES=16
W&B key loaded: True
PyTorch version: 2.11.0+cu130
CUDA available: True


In [3]:
# Setup paths for dataset
drive_path = '/content/drive/MyDrive/datasets/ucf50'
local_path = '/content'
url = 'https://www.crcv.ucf.edu/data/UCF50/UCF50.rar'
extract_path = os.path.join(local_path, 'datasets')
raw_dir = os.path.join(extract_path, 'UCF50')
frame_dir = os.path.join(extract_path,'UCF50_frames')

In [4]:
!python scripts/prep_dataset.py --drive_path {drive_path} --local_path {local_path} --url {url}

Local rar file already exists. Skipping download.
Extract path already exists at /content/datasets/UCF50. Skipping extraction.


In [5]:
video_counts = {}
frame_counts = []
failed_videos = []
for category in os.listdir('/content/datasets/UCF50'):
    category_path = os.path.join('/content/datasets/UCF50', category)
    video_count = 0
    for video in os.listdir(category_path):
        video_path = os.path.join(category_path, video)
        v_cap = cv2.VideoCapture(video_path)
        if not v_cap.isOpened():
            print("Failed to open video:", video_path)
            failed_videos.append(video_path)
            v_cap.release()
            continue
        v_len = int(v_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        v_cap.release()
        frame_counts.append(v_len)
        video_count += 1
    video_counts[category] = video_count
largest_category = max(video_counts, key=video_counts.get)
smallest_category = min(video_counts, key=video_counts.get)
print("Number of videos:", sum(video_counts.values()))
print("Largest category:", largest_category, video_counts[largest_category])
print("Smallest category:", smallest_category, video_counts[smallest_category])
print("Median frame count:", np.median(frame_counts))
print("Min frame count:", np.min(frame_counts))
print("Max frame count:", np.max(frame_counts))
print("Failed videos:", failed_videos)

Number of videos: 6681
Largest category: HorseRiding 197
Smallest category: PlayingViolin 100
Median frame count: 181.0
Min frame count: 29
Max frame count: 900
Failed videos: []


In [6]:
n_frames = 16
!python scripts/extract_frames.py --raw_dir {raw_dir} --n_frames {n_frames} --frame_dir {frame_dir}

Converting from avi: 100% 6681/6681 [00:00<00:00, 116229.13file/s]
Extracted 0 and skipped 6681 out of 6681 avis


In [7]:
!python -m pytest -v -rs tests/

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/assignments/Video_action_recognition
configfile: pytest.ini
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.5
collected 13 items                                                             

tests/test_dataset.py::test_raw_class_folders PASSED                     [  7%]
tests/test_dataset.py::test_video_avis PASSED                            [ 15%]
tests/test_dataset.py::test_ucf50_frames_dir PASSED                      [ 23%]
tests/test_dataset.py::test_video_dirs PASSED                            [ 30%]
tests/test_dataset.py::test_frame_counts PASSED                          [ 38%]
tests/test_dataset.py::test_frames_read PASSED                           [ 46%]
tests/test_models.py::test_batch_independence PASSED                     [ 53%]
te

In [8]:
!bash scripts/train.sh --cnn_backbone resnet50 --dropout 0.3 --num_workers 7 --run_name r50v2_do03

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 234MB/s]
Loading video dataset....
100% 50/50 [00:00<00:00, 5611.26it/s]
Splitting train/validation/test datasets....
Dropping 63 videos in groups outside 1-25
Train groups: [ 1  2  3  4  7  9 10 12 13 14 15 16 17 18 19 21 22 23 24]
Validation groups: [ 6 20]
Test groups: [ 5  8 11 25]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ttl074 (ttl074-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 90zhmnut (0.4s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/drive/MyDrive/assignments/Video_action_re

In [9]:
!bash scripts/test.sh models/r50v2_do03/best_model_wts.pt --cnn_backbone resnet50 --run_name test_r50v2_do03

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ttl074 (ttl074-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/drive/MyDrive/assignments/Video_action_recognition/wandb/run-20261025_193639-0aueb2kz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run test_r50v2_do03
wandb: ⭐️ View project at https://wandb.ai/ttl074-johns-hopkins-university/Video_Action_Recognition
wandb: 🚀 View run at https://wandb.ai/ttl074-johns-hopkins-university/Video_Action_Recognition/runs/0aueb2kz
100% 135/135 [00:52<00:00,  2.59it/s]
Loading video dataset....
100% 50/50 [00:00<00:00, 6158.49it/s]
The overall test accuracy is 89.2493%.
The overall test loss is 0.5015
{'BaseballPitch': 